# PEFT finetuning a Gen-AI model for Dialogue Summarization

The objective of this notebook is to fine-tune a pre-trained LLM model (FLAN-T5) to improve its performance on the dialog summarization task. We use Hugging face transformer library and dialog summary dataset. We fine-tune the pre-trained model using PEFT (Huggingface library) and then evaulate its performance compared to a reference fully fine-tuned model on the same task. For evaulation, we use ROUGE metric.

## Common installs

In [ ]:
#mount gdrive
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [ ]:
# Install the latest compatible torch and torchdata
%pip install --no-deps torch==2.3.0 torchdata==0.7.1 --quiet

# Install other updated packages (latest stable versions)
%pip install -U \
    datasets==2.19.1 \
    transformers==4.41.2 \
    peft==0.11.1 --quiet

%pip install evaluate --quiet
%pip install rouge_score --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.2/779.2 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 86.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 118.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 97.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 113.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 86.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7

In [ ]:
from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, GenerationConfig, TrainingArguments, Trainer
import torch
import time
import evaluate
import pandas as pd
import numpy as np

import os.path

CONST = {
  'DATASET_NAME' : 'knkarthick/dialogsum',
  'MODEL_NAME' : 'google/flan-t5-base',
  'INSTRUCT_MODEL_PATH' : 'truocpham/flan-dialogue-summary-checkpoint',
  'PEFT_CHECKPOINT_PATH' : '/content/drive/MyDrive/Colab Notebooks/Gen AI Notebooks/summary-model/checkpoints/peft-dialogue-summary-checkpoint/',
  'SUMMARY_RESULT_FILE': '/content/drive/MyDrive/Colab Notebooks/Gen AI Notebooks/summary-model/data/dialogue-summary-training-results.csv'
}

if not os.path.exists(CONST['PEFT_CHECKPOINT_PATH']):
    os.makedirs(CONST['PEFT_CHECKPOINT_PATH'])

PEFT_LOCAL_CHECKPOINT_FOUND = os.path.exists(CONST['PEFT_CHECKPOINT_PATH'] + 'adapter_config.json')

#set random seed

def set_seeds(seed=123):
  os.environ['PYTHONHASHSEED']=str(seed)
  torch.manual_seed(seed)
  torch.cuda.manual_seed(seed)
  torch.cuda.manual_seed_all(seed)
  np.random.seed(seed)

set_seeds()

rouge = evaluate.load('rouge')

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## Load dataset and model

In [ ]:
dataset = load_dataset(CONST['DATASET_NAME'])
dataset

Generating train split:   0%|          | 0/12460 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1500
    })
})

In [ ]:
model_name = CONST['MODEL_NAME']
original_model = AutoModelForSeq2SeqLM.from_pretrained(model_name, torch_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

In [ ]:
def show_trainable_parameters_details(model):
    trainable_params_cnt = 0
    total_params_cnt = 0

    for _, param in model.named_parameters():
        total_params_cnt += param.numel()
        if param.requires_grad:
            trainable_params_cnt += param.numel()

    return f"trainable parameters: {trainable_params_cnt}\n total parameters: {total_params_cnt}\n % of trainable model parameters: {100 * trainable_params_cnt / total_params_cnt:.2f}%"

print(show_trainable_parameters_details(original_model))

trainable parameters: 247577856
 total parameters: 247577856
 % of trainable model parameters: 100.00%


In [ ]:
#helper function to generate summary prompt given the dialog
def get_summary_prompt(dialogue):
  prompt = f"""
  Summarize the following conversation.

  {dialogue}

  Summary:
  """
  return prompt

## Test Original model with zero-shot inferencing

In [ ]:
index = 200

dialogue = dataset['test'][index]['dialogue']
summary = dataset['test'][index]['summary']

prompt = get_summary_prompt(dialogue)

inputs = tokenizer(prompt, return_tensors='pt')
output = tokenizer.decode(
    original_model.generate(
        inputs["input_ids"],
        max_new_tokens=200,
    )[0],
    skip_special_tokens=True
)

dash_line = '-'.join('' for x in range(100))
print(dash_line)
print(f'INPUT PROMPT:\n{prompt}')
print(dash_line)
print(f'BASELINE HUMAN SUMMARY:\n{summary}\n')
print(dash_line)
print(f'MODEL GENERATION - ZERO SHOT:\n{output}')

---------------------------------------------------------------------------------------------------
INPUT PROMPT:

  Summarize the following conversation.

  #Person1#: Have you considered upgrading your system?
#Person2#: Yes, but I'm not sure what exactly I would need.
#Person1#: You could consider adding a painting program to your software. It would allow you to make up your own flyers and banners for advertising.
#Person2#: That would be a definite bonus.
#Person1#: You might also want to upgrade your hardware because it is pretty outdated now.
#Person2#: How can we do that?
#Person1#: You'd probably need a faster processor, to begin with. And you also need a more powerful hard disc, more memory and a faster modem. Do you have a CD-ROM drive?
#Person2#: No.
#Person1#: Then you might want to add a CD-ROM drive too, because most new software programs are coming out on Cds.
#Person2#: That sounds great. Thanks.

  Summary:
  
-----------------------------------------------------------

In [ ]:
'''
helper function to tokenize prompt and reference summaries. returns input_ids and labels
'''
def tokenize_function(batch):
    prompt = [get_summary_prompt(dialogue) for dialogue in batch["dialogue"]]
    batch['input_ids'] = tokenizer(prompt, padding="max_length", truncation=True, return_tensors="pt").input_ids
    batch['labels'] = tokenizer(batch["summary"], padding="max_length", truncation=True, return_tensors="pt").input_ids

    return batch

# The tokenize_function code is handling all data across all splits in batches.
tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(['id', 'topic', 'dialogue', 'summary',])

Map:   0%|          | 0/12460 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

In [ ]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 500
    })
    test: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 1500
    })
})

## Load full fine-tuned instruct model (for reference)

In [ ]:
instruct_model = AutoModelForSeq2SeqLM.from_pretrained(CONST['INSTRUCT_MODEL_PATH'], torch_dtype=torch.bfloat16)

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

We already have summary results from various models stored for reference. We can use it to generate rouge scores for summaries produced by the original model and the instruct model.

In [ ]:
results = pd.read_csv(CONST['SUMMARY_RESULT_FILE'])

human_baseline_summaries = results['human_baseline_summaries'].values
original_model_summaries = results['original_model_summaries'].values
instruct_model_summaries = results['instruct_model_summaries'].values

original_model_results = rouge.compute(
    predictions=original_model_summaries,
    references=human_baseline_summaries[0:len(original_model_summaries)],
    use_aggregator=True,
    use_stemmer=True,
)

instruct_model_results = rouge.compute(
    predictions=instruct_model_summaries,
    references=human_baseline_summaries[0:len(instruct_model_summaries)],
    use_aggregator=True,
    use_stemmer=True,
)

print('ORIGINAL MODEL:')
print(original_model_results)
print('INSTRUCT MODEL:')
print(instruct_model_results)

ORIGINAL MODEL:
{'rouge1': np.float64(0.2334391967410257), 'rouge2': np.float64(0.0760552786391262), 'rougeL': np.float64(0.20142495235336488), 'rougeLsum': np.float64(0.2016464968104808)}
INSTRUCT MODEL:
{'rouge1': np.float64(0.42158467474716244), 'rouge2': np.float64(0.18033986205473646), 'rougeL': np.float64(0.33836289974974887), 'rougeLsum': np.float64(0.33829559790912717)}


## Parameter Efficient Fine-Tuning (PEFT)

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType
from peft import PeftModel, PeftConfig

'''
Function to simply load peft model from local checkpoint
'''
def load_pref_model_from_local_checkpoint():
  peft_model_base = AutoModelForSeq2SeqLM.from_pretrained(model_name, torch_dtype=torch.bfloat16)
  tokenizer = AutoTokenizer.from_pretrained(model_name)

  return PeftModel.from_pretrained(peft_model_base,
                                        CONST['PEFT_CHECKPOINT_PATH'],
                                        torch_dtype=torch.bfloat16,
                                        is_trainable=False)

'''
Function to train and save peft model
'''
def train_and_save_peft_model():

  #initialize adapter config
  lora_config = LoraConfig(
      r=32, # rank
      lora_alpha=32, #scaling
      target_modules=["q", "v"], #query and value matrices of attention modules
      lora_dropout=0.05,
      bias="none",
      task_type=TaskType.SEQ_2_SEQ_LM #
  )

  #get peft model from the original model and lora config. Note that the original model weights are frozen.
  peft_model = get_peft_model(original_model, lora_config)

  output_dir = f'./peft-dialogue-summary-training-{str(int(time.time()))}'

  peft_training_args = TrainingArguments(
      output_dir=output_dir,
      auto_find_batch_size=True,
      learning_rate=1e-3, # higher learning rate than full fine-tuning.
      num_train_epochs=1,
      logging_steps=1,
      max_steps=1,
      report_to='none'
  )

  #train the peft model on the train set
  peft_trainer = Trainer(
      model=peft_model,
      args=peft_training_args,
      train_dataset=tokenized_datasets["train"]
  )
  peft_trainer.train()

  #save the local checkpoint
  peft_trainer.model.save_pretrained(CONST['PEFT_CHECKPOINT_PATH'])
  tokenizer.save_pretrained(CONST['PEFT_CHECKPOINT_PATH'])
  return peft_model


In [ ]:

if PEFT_LOCAL_CHECKPOINT_FOUND:
  #load peft model from local checkpoint
  peft_model = load_pref_model_from_local_checkpoint()

else:
  #train and save peft model
  peft_model = train_and_save_peft_model()

print('peft model parameters:', show_trainable_parameters_details(peft_model))

peft model parameters: trainable parameters: 0
 total parameters: 251116800
 % of trainable model parameters: 0.00%


## Evaluate the Model Qualitatively

In [ ]:
index = 200
dialogue = dataset['test'][index]['dialogue']
human_baseline_summary = dataset['test'][index]['summary']

prompt = get_summary_prompt(dialogue)

input_ids = tokenizer(prompt, return_tensors="pt").input_ids

original_model_outputs = original_model.generate(input_ids=input_ids, generation_config=GenerationConfig(max_new_tokens=200, num_beams=1))
original_model_text_output = tokenizer.decode(original_model_outputs[0], skip_special_tokens=True)

instruct_model_outputs = instruct_model.generate(input_ids=input_ids, generation_config=GenerationConfig(max_new_tokens=200, num_beams=1))
instruct_model_text_output = tokenizer.decode(instruct_model_outputs[0], skip_special_tokens=True)

peft_model_outputs = peft_model.generate(input_ids=input_ids, generation_config=GenerationConfig(max_new_tokens=200, num_beams=1))
peft_model_text_output = tokenizer.decode(peft_model_outputs[0], skip_special_tokens=True)

print(dash_line)
print(f'BASELINE HUMAN SUMMARY:\n{human_baseline_summary}')
print(dash_line)
print(f'ORIGINAL MODEL:\n{original_model_text_output}')
print(dash_line)
print(f'INSTRUCT MODEL:\n{instruct_model_text_output}')
print(dash_line)
print(f'PEFT MODEL: {peft_model_text_output}')

---------------------------------------------------------------------------------------------------
BASELINE HUMAN SUMMARY:
#Person1# teaches #Person2# how to upgrade software and hardware in #Person2#'s system.
---------------------------------------------------------------------------------------------------
ORIGINAL MODEL:
#Person1#: I'm thinking of upgrading my computer.one#: I'm not sure what exactly I would need. #Person1#: I'd probably need a painting program. #Person2#: I'd probably need a faster processor. #Person1#: I'd probably need a faster hard disc. #Person2#: I'd probably need a CD-ROM drive.one#: I'd probably need a CD-ROM drive too.one#: I'd probably need a CD-ROM drive too.one#: I'd probably need a CD-ROM drive too.one#: I'd probably need a CD-ROM drive too.one#: I'd probably need a CD-ROM drive too.one#: I'd probably need a CD-ROM drive too.
---------------------------------------------------------------------------------------------------
INSTRUCT MODEL:
#Person1# s

## Evaluate the Model Quantitatively (ROUGE Metric)


In [ ]:
dialogues = dataset['test'][0:10]['dialogue']
human_baseline_summaries = dataset['test'][0:10]['summary']

original_model_summaries = []
instruct_model_summaries = []
peft_model_summaries = []

for idx, dialogue in enumerate(dialogues):

    prompt = get_summary_prompt(dialogue)

    input_ids = tokenizer(prompt, return_tensors="pt").input_ids

    human_baseline_text_output = human_baseline_summaries[idx]

    original_model_outputs = original_model.generate(input_ids=input_ids, generation_config=GenerationConfig(max_new_tokens=200))
    original_model_text_output = tokenizer.decode(original_model_outputs[0], skip_special_tokens=True)

    instruct_model_outputs = instruct_model.generate(input_ids=input_ids, generation_config=GenerationConfig(max_new_tokens=200))
    instruct_model_text_output = tokenizer.decode(instruct_model_outputs[0], skip_special_tokens=True)

    peft_model_outputs = peft_model.generate(input_ids=input_ids, generation_config=GenerationConfig(max_new_tokens=200))
    peft_model_text_output = tokenizer.decode(peft_model_outputs[0], skip_special_tokens=True)

    original_model_summaries.append(original_model_text_output)
    instruct_model_summaries.append(instruct_model_text_output)
    peft_model_summaries.append(peft_model_text_output)

zipped_summaries = list(zip(human_baseline_summaries, original_model_summaries, instruct_model_summaries, peft_model_summaries))

df = pd.DataFrame(zipped_summaries, columns = ['human_baseline_summaries', 'original_model_summaries', 'instruct_model_summaries', 'peft_model_summaries'])
df

,human_baseline_summaries,original_model_summaries,instruct_model_summaries,peft_model_summaries
0,Ms. Dawson helps #Person1# to write a memo to ...,#Person1#: I need to take a dictation for you....,#Person1# asks Ms. Dawson to take a dictation ...,#Person1# asks Ms. Dawson to take a dictation ...
1,In order to prevent employees from wasting tim...,#Person1#: I need to take a dictation for you....,#Person1# asks Ms. Dawson to take a dictation ...,#Person1# asks Ms. Dawson to take a dictation ...
2,Ms. Dawson takes a dictation for #Person1# abo...,#Person1#: I need to take a dictation for you....,#Person1# asks Ms. Dawson to take a dictation ...,#Person1# asks Ms. Dawson to take a dictation ...
3,#Person2# arrives late because of traffic jam....,The traffic jam at the Carrefour intersection ...,#Person2# got stuck in traffic again. #Person1...,#Person2# got stuck in traffic and #Person1# s...
4,#Person2# decides to follow #Person1#'s sugges...,The traffic jam at the Carrefour intersection ...,#Person2# got stuck in traffic again. #Person1...,#Person2# got stuck in traffic and #Person1# s...
5,#Person2# complains to #Person1# about the tra...,The traffic jam at the Carrefour intersection ...,#Person2# got stuck in traffic again. #Person1...,#Person2# got stuck in traffic and #Person1# s...
6,#Person1# tells Kate that Masha and Hero get d...,Masha and Hero are getting divorced..............,Masha and Hero are getting divorced. Kate can'...,Kate tells #Person2# Masha and Hero are gettin...
7,#Person1# tells Kate that Masha and Hero are g...,Masha and Hero are getting divorced..............,Masha and Hero are getting divorced. Kate can'...,Kate tells #Person2# Masha and Hero are gettin...
8,#Person1# and Kate talk about the divorce betw...,Masha and Hero are getting divorced..............,Masha and Hero are getting divorced. Kate can'...,Kate tells #Person2# Masha and Hero are gettin...
9,#Person1# and Brian are at the birthday party ...,"#Person1#: Happy birthday, Brian. #Person2#: T...",Brian's birthday is coming. #Person1# invites ...,Brian remembers his birthday and invites #Pers...


In [ ]:
rouge = evaluate.load('rouge')

original_model_results = rouge.compute(
    predictions=original_model_summaries,
    references=human_baseline_summaries[0:len(original_model_summaries)],
    use_aggregator=True,
    use_stemmer=True,
)

instruct_model_results = rouge.compute(
    predictions=instruct_model_summaries,
    references=human_baseline_summaries[0:len(instruct_model_summaries)],
    use_aggregator=True,
    use_stemmer=True,
)

peft_model_results = rouge.compute(
    predictions=peft_model_summaries,
    references=human_baseline_summaries[0:len(peft_model_summaries)],
    use_aggregator=True,
    use_stemmer=True,
)

print('ORIGINAL MODEL:')
print(original_model_results)
print('INSTRUCT MODEL:')
print(instruct_model_results)
print('PEFT MODEL:')
print(peft_model_results)

ORIGINAL MODEL:
{'rouge1': np.float64(0.19300148115773114), 'rouge2': np.float64(0.08700115442156761), 'rougeL': np.float64(0.17084923300383825), 'rougeLsum': np.float64(0.16894512894512892)}
INSTRUCT MODEL:
{'rouge1': np.float64(0.3973035592160016), 'rouge2': np.float64(0.17137215542557926), 'rougeL': np.float64(0.28172493028301904), 'rougeLsum': np.float64(0.2824424656379544)}
PEFT MODEL:
{'rouge1': np.float64(0.3714845752656546), 'rouge2': np.float64(0.12020504055478767), 'rougeL': np.float64(0.2766553571288646), 'rougeLsum': np.float64(0.276695852393251)}


Note that the PEFT results are somewhat inferior compared to instruct model (full finetuning), however much better compared to the original pre-trained model.

The `data/dialogue-summary-training-results.csv' file already contains the summaries produced by all models. Because of the computing constraints, we just compute rouge scores on the summaries from the csv file and report it here.

In [ ]:
human_baseline_summaries = results['human_baseline_summaries'].values
original_model_summaries = results['original_model_summaries'].values
instruct_model_summaries = results['instruct_model_summaries'].values
peft_model_summaries     = results['peft_model_summaries'].values

original_model_results = rouge.compute(
    predictions=original_model_summaries,
    references=human_baseline_summaries[0:len(original_model_summaries)],
    use_aggregator=True,
    use_stemmer=True,
)

instruct_model_results = rouge.compute(
    predictions=instruct_model_summaries,
    references=human_baseline_summaries[0:len(instruct_model_summaries)],
    use_aggregator=True,
    use_stemmer=True,
)

peft_model_results = rouge.compute(
    predictions=peft_model_summaries,
    references=human_baseline_summaries[0:len(peft_model_summaries)],
    use_aggregator=True,
    use_stemmer=True,
)

print('ORIGINAL MODEL:')
print(original_model_results)
print('INSTRUCT MODEL:')
print(instruct_model_results)
print('PEFT MODEL:')
print(peft_model_results)

ORIGINAL MODEL:
{'rouge1': np.float64(0.2334391967410257), 'rouge2': np.float64(0.0760552786391262), 'rougeL': np.float64(0.20142495235336488), 'rougeLsum': np.float64(0.2016464968104808)}
INSTRUCT MODEL:
{'rouge1': np.float64(0.42158467474716244), 'rouge2': np.float64(0.18033986205473646), 'rougeL': np.float64(0.33836289974974887), 'rougeLsum': np.float64(0.33829559790912717)}
PEFT MODEL:
{'rouge1': np.float64(0.40807759801703863), 'rouge2': np.float64(0.16324229963604536), 'rougeL': np.float64(0.3251941061148622), 'rougeLsum': np.float64(0.3251390734179663)}


The results clearly show that PEFT exhibits somewhat lower performance than the fully fine-tuned model. However, the benefits of PEFT generally outweigh the slightly-lower performance.

Calculate the improvement of PEFT over the original model:

In [ ]:
print("Absolute percentage improvement of PEFT MODEL over ORIGINAL MODEL")

improvement = (np.array(list(peft_model_results.values())) - np.array(list(original_model_results.values())))
for key, value in zip(peft_model_results.keys(), improvement):
    print(f'{key}: {value*100:.2f}%')

Absolute percentage improvement of PEFT MODEL over ORIGINAL MODEL
rouge1: 17.46%
rouge2: 8.72%
rougeL: 12.38%
rougeLsum: 12.35%


Calculate the (negative) improvement of PEFT over a full fine-tuned model:

In [ ]:
print("Absolute percentage improvement of PEFT MODEL over INSTRUCT MODEL")

improvement = (np.array(list(peft_model_results.values())) - np.array(list(instruct_model_results.values())))
for key, value in zip(peft_model_results.keys(), improvement):
    print(f'{key}: {value*100:.2f}%')

Absolute percentage improvement of PEFT MODEL over INSTRUCT MODEL
rouge1: -1.35%
rouge2: -1.71%
rougeL: -1.32%
rougeLsum: -1.32%
